In [1]:
import os
import json
import numpy as np
import cv2
from PIL import Image

def natural_sort_key(s):
    """Key function for natural sorting of filenames"""
    import re
    return [int(text) if text.isdigit() else text.lower() 
            for text in re.split('([0-9]+)', s)]

In [2]:
def process_directory(input_dir, output_dir='building_damage_segments', output_size=(128, 128), pad=5):
    """Process all image/JSON pairs in a directory"""
    
    os.makedirs(output_dir, exist_ok=True)
    damage_types = ['major-damage', 'minor-damage', 'unknown']
    for dtype in damage_types:
        os.makedirs(os.path.join(output_dir, dtype), exist_ok=True)

    all_files = sorted(os.listdir(input_dir), key=natural_sort_key)
    
    processed_pairs = 0
    for filename in all_files:
        if filename.endswith('.png'):
            base_name = filename[:-4]
            json_file = f"{base_name}.json"
            
            if json_file in all_files:
                image_path = os.path.join(input_dir, filename)
                json_path = os.path.join(input_dir, json_file)
                
                print(f"Processing pair: {filename} + {json_file}")
                process_single_pair(image_path, json_path, output_dir, output_size, pad)
                processed_pairs += 1
    
    print(f"\nCompleted! Processed {processed_pairs} image/JSON pairs")

In [3]:
def process_single_pair(image_path, json_path, output_dir, output_size=(128, 128), pad=5):
    """Process one image/JSON pair with padding, resizing, and mask saving"""
    try:
        image = np.array(Image.open(image_path))
        with open(json_path) as f:
            data = json.load(f)

        base_name = os.path.splitext(os.path.basename(image_path))[0]

        def parse_wkt_polygon(wkt_str):
            coords_str = wkt_str.split('((')[1].split('))')[0]
            return [tuple(map(float, pair.split())) for pair in coords_str.split(', ')]

        for feature in data['features']['xy']:
            properties = feature['properties']
            if properties['feature_type'] != 'building':
                continue

            uid = properties['uid']
            subtype = properties.get('subtype', 'unknown')
            subtype = subtype if subtype in ['major-damage', 'minor-damage'] else 'unknown'

            polygon_coords = parse_wkt_polygon(feature['wkt'])
            polygon_pts = np.array(polygon_coords, dtype=np.int32)

            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            cv2.fillPoly(mask, [polygon_pts], 1)

            y, x = np.where(mask)
            if len(x) == 0 or len(y) == 0:
                continue
            x1, y1 = max(np.min(x) - pad, 0), max(np.min(y) - pad, 0)
            x2, y2 = min(np.max(x) + pad, image.shape[1] - 1), min(np.max(y) + pad, image.shape[0] - 1)

            cropped = image[y1:y2+1, x1:x2+1]
            cropped_mask = mask[y1:y2+1, x1:x2+1]
            cropped_resized = cv2.resize(cropped, output_size)
            mask_resized = cv2.resize(cropped_mask, output_size, interpolation=cv2.INTER_NEAREST)

            crop_filename = f"{base_name}_{uid}_{subtype}.png"
            mask_filename = f"{base_name}_{uid}_{subtype}_mask.png"
            cv2.imwrite(os.path.join(output_dir, subtype, crop_filename), cv2.cvtColor(cropped_resized, cv2.COLOR_RGB2BGR))
            cv2.imwrite(os.path.join(output_dir, subtype, mask_filename), mask_resized * 255)

    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")

In [4]:
if __name__ == "__main__":
    input_directory = r"C:\Users\LOQ\Documents\results"  
    output_directory = "building_damage_segments"
    
    process_directory(input_directory, output_directory, output_size=(128, 128), pad=5)

Processing pair: hurricane-harvey_00000000_post_disaster.png + hurricane-harvey_00000000_post_disaster.json
Processing pair: hurricane-harvey_00000001_post_disaster.png + hurricane-harvey_00000001_post_disaster.json
Processing pair: hurricane-harvey_00000002_post_disaster.png + hurricane-harvey_00000002_post_disaster.json
Processing pair: hurricane-harvey_00000004_post_disaster.png + hurricane-harvey_00000004_post_disaster.json
Processing pair: hurricane-harvey_00000005_post_disaster.png + hurricane-harvey_00000005_post_disaster.json

Completed! Processed 5 image/JSON pairs
